In [1]:
from requests.exceptions import ReadTimeout
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import pandas as pd
import random
import time 
import json
import os

from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import (
    boxscoreadvancedv2,
    playergamelog,
    teamyearbyyearstats,
    commonteamroster
)

#---------------------------------------------------------------------------------
#define helper functions
def normalize_for_postgres(df):
    """Normalize DataFrame column names for PostgreSQL"""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df
    
def get_processed_players(engine, table_name='player_game_stats_temp'):
    """Get set of player IDs we've already processed"""
    inspector = inspect(engine)
    if table_name in inspector.get_table_names():
        query = f"SELECT DISTINCT PLAYER_ID FROM {table_name}"
        processed_df = pd.read_sql(query, engine)
        return set(processed_df['PLAYER_ID'].tolist())
    return set()

#---------------------------------------------------------------------------
#Variable setup 
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Season formats - NBA API uses different formats in different endpoints!
# seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20',
#            '2018-19', '2017-18', '2016-17', '2015-16', '2014-15',
#            '2013-14', '2012-13', '2011-12', '2010-11', '2009-10']
seasons = ['2024-25']

# For PlayerGameLog.SEASON_ID filtering (API internal format: '2YYYY')
# seasons_api = ['22023', '22022', '22021', '22020', '22019',
#                '22018', '22017', '22016', '22015', '22014',
#                '22013', '22012', '22011', '22010', '22009']
seasons_api = ['22024']

nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total")

#----------------------------------------------------------------------------
#get players based on team rosters in each year we are targeting 

all_players = []
team_count = 0
seen_player_ids = set()

for team in nba_teams:
    team_id = team['id']
    team_name = team['full_name']
    team_count += 1
    
    print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name} (ID: {team_id})")
    
    try:   
        for season in seasons:
                
            # Add a sleep to avoid rate limiting
            sleep_time = random.uniform(1.5, 3.0)
            print(f"  Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time)
            
            # Get team roster for that season (uses human-readable format)
            print(f"  Fetching {season} roster for {team_name}...")
            roster = commonteamroster.CommonTeamRoster(
                team_id=team_id,
                season=season
            )
            roster_df = roster.get_data_frames()[0]

            # Filter out players we've already seen
            # Convert to dict records
            # Update the set of seen player IDs
            new_players_df = roster_df[~roster_df['PLAYER_ID'].isin(seen_player_ids)]
            new_players = new_players_df.to_dict('records')
            seen_player_ids.update(new_players_df['PLAYER_ID'].tolist())
            
            player_count = len(new_players)
            all_players.extend(new_players)
            print(f"  ✓ Added {player_count} players from {team_name}")
        
    except Exception as e:
        print(f"  ⚠ Error processing {team_name}: {str(e)}")
    
    # Add a separator for readability
    print("-" * 50)
    
print("\nSummary:")
print(f"Processed {team_count} total teams")
print(f"Collected data for {len(all_players)} players")
print(len(all_players))

#------------------------------------------------------------------------------
#gather player games

# Get already processed players
print("Checking for existing progress...")
processed_player_ids = get_processed_players(engine, 'player_game_stats_temp')
print(f"Found {len(processed_player_ids)} players already processed")

# Filter out already-processed players
players_to_process = [p for p in all_players if p['PLAYER_ID'] not in processed_player_ids]
print(f"Remaining players to process: {len(players_to_process)}")

#setup dataframe
player_stats_df = pd.DataFrame()
i = 1

for player in players_to_process:
    while True:
        try:
            print(f"Getting stats for {player['PLAYER']}")
            #get player dict and ID
            PID = player['PLAYER_ID']
            
            #retreive player game stats
            game_log = playergamelog.PlayerGameLog(
                player_id=PID,
                season=SeasonAll.all
            )
            df = game_log.get_data_frames()[0]

            if not df.empty:
                # CORRECTED: Use seasons_api format for SEASON_ID filtering
                df = df[df['SEASON_ID'].isin(seasons_api)]
                if not df.empty:
                    player_stats_df = pd.concat([player_stats_df, df], ignore_index=True)
                    print(f"Data retreived for {player['PLAYER']} ({len(df)} games)")

                    # SAVE PROGRESS every 50 players
                    if i % 50 == 0:
                        print(f"\n  💾 Saving checkpoint at player {i}...")
                        player_stats_df.to_sql(
                            'player_game_stats_temp',
                            engine,
                            if_exists='append',
                            index=False
                        )
                        player_stats_df = pd.DataFrame()  # Clear memory
                        print("  ✓ Checkpoint saved\n")
                else:
                    print(f"No games in target seasons for {player['PLAYER']}")
            else:
                print(f"No games found for {player['PLAYER']}")
        
            if i % 40 == 0:
                time.sleep(round(random.uniform(60, 120), 1))
                print()
                print("Long Sleep!")
                print()
                
            else:
                time.sleep(round(random.uniform(3, 4), 1))
            i += 1

            #exit loop
            break 
            
        except (ReadTimeout, json.decoder.JSONDecodeError, Exception) as e:
            print(f"Error for {player['PLAYER']}: {e} - retrying after 60 seconds")
            time.sleep(180)
            continue 

# Save any remaining data
if not player_stats_df.empty:
    player_stats_df.to_sql('player_game_stats_temp', engine, if_exists='append', index=False)

# Load all collected player stats
print("\nLoading all collected player stats...")
all_player_stats = pd.read_sql('SELECT * FROM player_game_stats_temp', engine)


print(f"Total records: {len(all_player_stats)}")
all_player_stats = normalize_for_postgres(all_player_stats)

all_player_stats.to_sql(
    'player_game_stats',
    engine,
    if_exists='append',
    index=False
)

print("✅ Complete! All data saved to 'player_game_stats'")
print("="*60)

Found 30 teams in total
[1/30] Processing team: Atlanta Hawks (ID: 1610612737)
  Sleeping for 1.90 seconds...
  Fetching 2024-25 roster for Atlanta Hawks...
  ✓ Added 18 players from Atlanta Hawks
--------------------------------------------------
[2/30] Processing team: Boston Celtics (ID: 1610612738)
  Sleeping for 1.63 seconds...
  Fetching 2024-25 roster for Boston Celtics...
  ✓ Added 17 players from Boston Celtics
--------------------------------------------------
[3/30] Processing team: Cleveland Cavaliers (ID: 1610612739)
  Sleeping for 2.37 seconds...
  Fetching 2024-25 roster for Cleveland Cavaliers...
  ✓ Added 18 players from Cleveland Cavaliers
--------------------------------------------------
[4/30] Processing team: New Orleans Pelicans (ID: 1610612740)
  Sleeping for 1.65 seconds...
  Fetching 2024-25 roster for New Orleans Pelicans...
  ✓ Added 18 players from New Orleans Pelicans
--------------------------------------------------
[5/30] Processing team: Chicago Bulls 

ProgrammingError: (psycopg2.errors.UndefinedColumn) column "player_id" does not exist
LINE 1: SELECT DISTINCT PLAYER_ID FROM player_game_stats_temp
                        ^
HINT:  Perhaps you meant to reference the column "player_game_stats_temp.Player_ID".

[SQL: SELECT DISTINCT PLAYER_ID FROM player_game_stats_temp]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [9]:
# Run this to clear the 'failed' state
engine.connect().rollback()

In [10]:
from requests.exceptions import ReadTimeout
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import pandas as pd
import random
import time 
import json
import os

from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import (
    boxscoreadvancedv2,
    playergamelog,
    teamyearbyyearstats,
    commonteamroster
)

# Load all collected player stats
print("\nLoading all collected player stats...")
all_player_stats = pd.read_sql('SELECT * FROM player_game_stats_temp', engine)


print(f"Total records: {len(all_player_stats)}")
all_player_stats = normalize_for_postgres(all_player_stats)

all_player_stats.to_sql(
    'player_game_stats',
    engine,
    if_exists='append',
    index=False
)

print("✅ Complete! All data saved to 'player_game_stats'")
print("="*60)


Loading all collected player stats...
Total records: 25840
✅ Complete! All data saved to 'player_game_stats'


In [1]:
from requests.exceptions import ReadTimeout
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import pandas as pd
import random
import time 
import json
import os

from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import (
    boxscoreadvancedv2,
    playergamelog,
    teamyearbyyearstats,
    commonteamroster
)

#---------------------------------------------------------------------------------
#define helper functions
def get_processed_players(engine, table_name='player_game_stats_temp'):
    """Get set of player IDs we've already processed"""
    inspector = inspect(engine)
    if table_name in inspector.get_table_names():
        query = f"SELECT DISTINCT PLAYER_ID FROM {table_name}"
        processed_df = pd.read_sql(query, engine)
        return set(processed_df['PLAYER_ID'].tolist())
    return set()

# def get_processed_games(engine, table_name='player_game_stats'):
#     """Get set of game IDs we've already fetched advanced stats for"""
#     inspector = inspect(engine)
#     if table_name in inspector.get_table_names():
#         # Check if the advanced stats columns exist (indicating we've processed this game)
#         query = f"SELECT DISTINCT GAME_ID FROM {table_name} WHERE E_OFF_RATING IS NOT NULL"
#         processed_df = pd.read_sql(query, engine)
#         return set(processed_df['GAME_ID'].tolist())
#     return set()

# def chunks(lst, n):
#     """Split list into chunks of size n"""
#     for i in range(0, len(lst), n):
#         yield lst[i:i + n]

#---------------------------------------------------------------------------
#Variable setup 
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Season formats - NBA API uses different formats in different endpoints!
# For TeamYearByYearStats.YEAR and CommonTeamRoster.season parameter (human-readable format)
seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20',
           '2018-19', '2017-18', '2016-17', '2015-16', '2014-15',
           '2013-14', '2012-13', '2011-12', '2010-11', '2009-10']

# For PlayerGameLog.SEASON_ID filtering (API internal format: '2YYYY')
seasons_api = ['22023', '22022', '22021', '22020', '22019',
               '22018', '22017', '22016', '22015', '22014',
               '22013', '22012', '22011', '22010', '22009']

nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total")

#----------------------------------------------------------------------------
#get players based on team rosters in each year we are targeting 

all_players = []
team_count = 0
seen_player_ids = set()

for team in nba_teams:
    team_id = team['id']
    team_name = team['full_name']
    team_count += 1
    
    print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name} (ID: {team_id})")
    
    try:
        # Check if the team was active that season
        # print(f"  Fetching season history for {team_name}...")
        # team_seasons = teamyearbyyearstats.TeamYearByYearStats(team_id=team_id)
        # seasons_df = team_seasons.get_data_frames()[0]
        
        # # Add a sleep to avoid rate limiting
        # sleep_time = random.uniform(1.5, 3.0)
        # print(f"  Sleeping for {sleep_time:.2f} seconds...")
        # time.sleep(sleep_time)
        
        for season in seasons:
                
            # Add a sleep to avoid rate limiting
            sleep_time = random.uniform(1.5, 3.0)
            print(f"  Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time)
            
            # Get team roster for that season (uses human-readable format)
            print(f"  Fetching {season} roster for {team_name}...")
            roster = commonteamroster.CommonTeamRoster(
                team_id=team_id,
                season=season
            )
            roster_df = roster.get_data_frames()[0]

            # Filter out players we've already seen
            # Convert to dict records
            # Update the set of seen player IDs
            new_players_df = roster_df[~roster_df['PLAYER_ID'].isin(seen_player_ids)]
            new_players = new_players_df.to_dict('records')
            seen_player_ids.update(new_players_df['PLAYER_ID'].tolist())
            
            player_count = len(new_players)
            all_players.extend(new_players)
            print(f"  ✓ Added {player_count} players from {team_name}")
        
    except Exception as e:
        print(f"  ⚠ Error processing {team_name}: {str(e)}")
    
    # Add a separator for readability
    print("-" * 50)
    
print("\nSummary:")
print(f"Processed {team_count} total teams")
print(f"Collected data for {len(all_players)} players")
print(len(all_players))

#------------------------------------------------------------------------------
#gather player games

# Get already processed players
print("Checking for existing progress...")
processed_player_ids = get_processed_players(engine, 'player_game_stats_temp')
print(f"Found {len(processed_player_ids)} players already processed")

# Filter out already-processed players
players_to_process = [p for p in all_players if p['PLAYER_ID'] not in processed_player_ids]
print(f"Remaining players to process: {len(players_to_process)}")

#setup dataframe
player_stats_df = pd.DataFrame()
i = 1

for player in players_to_process:
    while True:
        try:
            print(f"Getting stats for {player['PLAYER']}")
            #get player dict and ID
            PID = player['PLAYER_ID']
            
            #retreive player game stats
            game_log = playergamelog.PlayerGameLog(
                player_id=PID,
                season=SeasonAll.all
            )
            df = game_log.get_data_frames()[0]

            if not df.empty:
                # CORRECTED: Use seasons_api format for SEASON_ID filtering
                df = df[df['SEASON_ID'].isin(seasons_api)]
                if not df.empty:
                    player_stats_df = pd.concat([player_stats_df, df], ignore_index=True)
                    print(f"Data retreived for {player['PLAYER']} ({len(df)} games)")

                    # SAVE PROGRESS every 50 players
                    if i % 50 == 0:
                        print(f"\n  💾 Saving checkpoint at player {i}...")
                        player_stats_df.to_sql(
                            'player_game_stats_temp',
                            engine,
                            if_exists='append',
                            index=False
                        )
                        player_stats_df = pd.DataFrame()  # Clear memory
                        print("  ✓ Checkpoint saved\n")
                else:
                    print(f"No games in target seasons for {player['PLAYER']}")
            else:
                print(f"No games found for {player['PLAYER']}")
        
            if i % 40 == 0:
                time.sleep(round(random.uniform(60, 120), 1))
                print()
                print("Long Sleep!")
                print()
                
            else:
                time.sleep(round(random.uniform(3, 4), 1))
            i += 1

            #exit loop
            break 
            
        except (ReadTimeout, json.decoder.JSONDecodeError, Exception) as e:
            print(f"Error for {player['PLAYER']}: {e} - retrying after 60 seconds")
            time.sleep(180)
            continue 

# Save any remaining data
if not player_stats_df.empty:
    player_stats_df.to_sql('player_game_stats_temp', engine, if_exists='append', index=False)

# Load all collected player stats
print("\nLoading all collected player stats...")
all_player_stats = pd.read_sql('SELECT * FROM player_game_stats_temp', engine)


print(f"Total records: {len(all_player_stats)}")

all_player_stats.to_sql(
    'player_game_stats',
    engine,
    if_exists='replace',
    index=False
)

print("✅ Complete! All data saved to 'player_game_stats'")
print("="*60)

#-------------------------------------------------------------------------------------
#Get UNIQUE game IDs from collected data

# Extract ALL unique game IDs from the complete dataset
# all_game_ids = set(all_player_stats['GAME_ID'].unique())

# print("\nChecking which games already have advanced stats...")
# processed_game_ids = get_processed_games(engine, 'player_game_stats')
# games_to_process = list(all_game_ids - processed_game_ids)

# print(f"  Total games needed: {len(all_game_ids)}")
# print(f"  Already processed: {len(processed_game_ids)}")
# print(f"  Remaining to fetch: {len(games_to_process)}")

# if len(games_to_process) == 0:
#     print("\n✅ All games already have advanced stats! Nothing to do.")
#     exit()

# #-------------------------------------------------------------------------------------
# # define functions for batching game data and saving progress
# print("\nStep 4: Fetching advanced stats in batches...")

# game_batches = list(chunks(games_to_process, 100))
# print(f"Processing {len(games_to_process)} games in {len(game_batches)} batches")

# #-------------------------------------------------------------------------------------
# # Fetch advanced stats in batches
# j = 1

# for batch_num, batch in enumerate(game_batches, 1):
#     print(f"\n--- Batch {batch_num}/{len(game_batches)} ---")
#     batch_advanced_stats = pd.DataFrame()
    
#     for game_id in batch:
#         while True:
#             try:
#                 print(f"Fetching advanced stats for game {j}/{len(games_to_process)}: {game_id}")
                
#                 advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=game_id)
#                 advanced_df = advanced.get_data_frames()[0]  # Player stats
#                 batch_advanced_stats = pd.concat([batch_advanced_stats, advanced_df], ignore_index=True)
                
#                 # Rate limiting
#                 if j % 100 == 0:
#                     time.sleep(random.uniform(60, 120))
#                 else:
#                     time.sleep(random.uniform(1, 2))
                
#                 j += 1
#                 break
                
#             except Exception as e:
#                 print(f"Error fetching game {game_id}: {e} - retrying")
#                 time.sleep(180)
#                 continue


#     # KEY CHANGE: Merge this batch and save to final table incrementally
#     print(f"\nMerging and saving batch {batch_num}...")
    
#     # Get the player stats for games in this batch
#     batch_game_ids = batch_advanced_stats['GAME_ID'].unique()
#     batch_player_stats = all_player_stats[all_player_stats['GAME_ID'].isin(batch_game_ids)]
    
#     # Merge basic and advanced for this batch
#     batch_combined = batch_player_stats.merge(
#         batch_advanced_stats,
#         on=['GAME_ID', 'PLAYER_ID'],
#         how='left',
#         suffixes=('', '_adv')
#     )
    
#     # Save to database (append after first batch)
#     batch_combined.to_sql(
#         'player_game_stats',
#         engine,
#         if_exists='append',
#         index=False
#     )
    
#     print(f"✓ Batch {batch_num} saved ({len(batch_combined)} records)")
#     print(f"  Progress: {j-1}/{len(games_to_process)} games complete")

Found 30 teams in total
[1/30] Processing team: Atlanta Hawks (ID: 1610612737)
  Sleeping for 2.81 seconds...
  Fetching 2023-24 roster for Atlanta Hawks...
  ✓ Added 18 players from Atlanta Hawks
  Sleeping for 2.44 seconds...
  Fetching 2022-23 roster for Atlanta Hawks...
  ✓ Added 4 players from Atlanta Hawks
  Sleeping for 1.90 seconds...
  Fetching 2021-22 roster for Atlanta Hawks...
  ✓ Added 10 players from Atlanta Hawks
  Sleeping for 2.58 seconds...


KeyboardInterrupt: 